# 📐 03 — Price Elasticity Analysis
**Dynamic Pricing Engine for Small Businesses**

**Objective:** Measure how sensitive demand is to price changes across different contexts.

**Price Elasticity of Demand (PED):**  
> `Elasticity = (% change in demand) / (% change in price)`

| Elasticity | Classification | Pricing Strategy |
|-----------|---------------|------------------|
| `e < -1`  | Elastic (price-sensitive) | Lower price to boost revenue |
| `-1 ≤ e < 0` | Inelastic (price-insensitive) | Small price increase is safe |
| `e ≥ 0`   | Unusual (Giffen/prestige) | Investigate further |

We will: compute rolling elasticity, visualize distribution, segment by season, and build a model-based elasticity heatmap.

## 0. Setup

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..')))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': '#0f0f0f', 'axes.facecolor': '#1a1a2e',
    'axes.edgecolor': '#444', 'axes.labelcolor': '#ddd',
    'xtick.color': '#aaa', 'ytick.color': '#aaa',
    'text.color': '#eee', 'grid.color': '#2a2a2a', 'grid.linestyle': '--',
})
PALETTE = ['#6C63FF', '#FF6B6B', '#FFD93D', '#6BCB77', '#4D96FF', '#FF922B']

from src.data_collection.data_simulator import simulate_sales
from src.models.demand_model import build_features, FEATURES
from src.models.elasticity import point_elasticity, elasticity_from_history, classify_elasticity

df = simulate_sales()
df = build_features(df)
print(f'Dataset ready: {df.shape}')
df.head(3)

## 1. Compute Rolling Elasticity

In [ ]:
df['elasticity'] = elasticity_from_history(df)

# Clip extreme outliers for visualization (keep 1st–99th percentile)
q1, q99 = df['elasticity'].quantile([0.01, 0.99])
df['elasticity_clipped'] = df['elasticity'].clip(q1, q99)

# Rolling 30-day average
df['elasticity_30d_ma'] = df['elasticity_clipped'].rolling(30, min_periods=1).mean()

# Classify each row
df['elasticity_class'] = df['elasticity_clipped'].apply(classify_elasticity)

avg_e = df['elasticity_clipped'].dropna().mean()
print(f'Average Elasticity: {avg_e:.4f}')
print(f'Classification    : {classify_elasticity(avg_e)}')
print(f'\nClass distribution:')
print(df['elasticity_class'].value_counts())

## 2. Elasticity Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Price Elasticity Distribution', fontsize=13, fontweight='bold')

# Histogram
axes[0].hist(df['elasticity_clipped'].dropna(), bins=40, color=PALETTE[0], edgecolor='black', alpha=0.85)
axes[0].axvline(-1,  color=PALETTE[1], lw=2, linestyle='--', label='Elastic boundary (e = -1)')
axes[0].axvline(0,   color=PALETTE[2], lw=2, linestyle='--', label='Zero elasticity')
axes[0].axvline(avg_e, color=PALETTE[3], lw=2, linestyle='-', label=f'Average e = {avg_e:.3f}')
axes[0].set_title('Elasticity Histogram')
axes[0].set_xlabel('Elasticity Coefficient'); axes[0].set_ylabel('Frequency')
axes[0].legend(fontsize=8)

# Class pie
class_counts = df['elasticity_class'].value_counts()
short_labels = []
for label in class_counts.index:
    if 'Elastic' in label and 'price-sensitive' in label: short_labels.append('Elastic')
    elif 'Inelastic' in label: short_labels.append('Inelastic')
    else: short_labels.append('Unusual')
axes[1].pie(class_counts.values, labels=short_labels,
    colors=PALETTE[:len(class_counts)], startangle=90,
    wedgeprops=dict(edgecolor='black'), textprops={'color':'#eee','fontsize':10},
    autopct='%1.1f%%', pctdistance=0.75)
axes[1].set_title('Elasticity Classification')

fig.patch.set_facecolor('#0f0f0f')
plt.tight_layout(); plt.show()

**Finding:** The majority of day-to-day elasticities cluster around the elastic/inelastic boundary. The average elasticity tells us the overall price sensitivity of the product category.

## 3. Elasticity Over Time

In [ ]:
fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(df['date'], df['elasticity_clipped'], color=PALETTE[0], alpha=0.35, lw=0.8, label='Daily elasticity')
ax.plot(df['date'], df['elasticity_30d_ma'], color=PALETTE[1], lw=2, label='30-day MA')
ax.axhline(-1, color=PALETTE[2], linestyle='--', lw=1.5, label='Elastic boundary (e = -1)')
ax.axhline(0,  color='white',    linestyle='--', lw=1, alpha=0.4, label='Zero')

# Shade festival periods
for _, row in df[df['is_festival'] == 1].iterrows():
    ax.axvspan(row['date'], row['date'] + pd.Timedelta(days=1), alpha=0.08, color=PALETTE[3])

ax.set_title('Price Elasticity Over Time (festivals shaded in green)', fontsize=12)
ax.set_xlabel('Date'); ax.set_ylabel('Elasticity Coefficient'); ax.legend(fontsize=9)
fig.patch.set_facecolor('#0f0f0f')
plt.tight_layout(); plt.show()

**Finding:** Elasticity fluctuates daily due to noise, but the 30-day moving average reveals the underlying trend. Look for clusters where elasticity dips below –1 — those are days to avoid price increases.

## 4. Elasticity by Season (Weekend vs Festival)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Elasticity by Seasonal Context', fontsize=13, fontweight='bold')

# Weekend segmentation
bp1 = axes[0].boxplot(
    [df[df['is_weekend']==0]['elasticity_clipped'].dropna(),
     df[df['is_weekend']==1]['elasticity_clipped'].dropna()],
    labels=['Weekday', 'Weekend'], patch_artist=True,
    boxprops=dict(color='white'), medianprops=dict(color=PALETTE[2], lw=2),
    whiskerprops=dict(color='white'), capprops=dict(color='white'),
    flierprops=dict(marker='o', alpha=0.2, markersize=3))
for patch, c in zip(bp1['boxes'], [PALETTE[0], PALETTE[1]]): patch.set_facecolor(c)
axes[0].axhline(-1, color=PALETTE[2], linestyle='--', lw=1, label='Elastic boundary')
axes[0].set_title('Elasticity: Weekday vs Weekend'); axes[0].set_ylabel('Elasticity')
axes[0].legend(fontsize=8)

# Festival segmentation
bp2 = axes[1].boxplot(
    [df[df['is_festival']==0]['elasticity_clipped'].dropna(),
     df[df['is_festival']==1]['elasticity_clipped'].dropna()],
    labels=['Normal Day', 'Festival'], patch_artist=True,
    boxprops=dict(color='white'), medianprops=dict(color=PALETTE[2], lw=2),
    whiskerprops=dict(color='white'), capprops=dict(color='white'),
    flierprops=dict(marker='o', alpha=0.2, markersize=3))
for patch, c in zip(bp2['boxes'], [PALETTE[0], PALETTE[3]]): patch.set_facecolor(c)
axes[1].axhline(-1, color=PALETTE[2], linestyle='--', lw=1, label='Elastic boundary')
axes[1].set_title('Elasticity: Normal vs Festival'); axes[1].set_ylabel('Elasticity')
axes[1].legend(fontsize=8)

fig.patch.set_facecolor('#0f0f0f')
plt.tight_layout(); plt.show()

print('Median elasticity by context:')
for ctx, mask in [('Weekday', df['is_weekend']==0), ('Weekend', df['is_weekend']==1),
                  ('Normal', df['is_festival']==0), ('Festival', df['is_festival']==1)]:
    e = df[mask]['elasticity_clipped'].dropna().median()
    print(f'  {ctx:10s}: {e:.4f}  → {classify_elasticity(e).split("—")[0].strip()}')

**Finding:** Festival periods show higher (less negative) elasticity — customers are willing to pay more during festivals despite price increases. This justifies price *increases* during festival seasons, not just static discounts.

## 5. Model-Based Elasticity — Price Sensitivity Heatmap

In [ ]:
import joblib
MODEL_PATH = Path('..') / 'data' / 'processed' / 'demand_model.pkl'

if MODEL_PATH.exists():
    model = joblib.load(MODEL_PATH)
    print(f'✅ Loaded trained model from {MODEL_PATH}')

    # Compute elasticity at different price/competitor_price combinations
    prices = np.linspace(80, 200, 15)
    comp_prices = np.linspace(80, 200, 15)
    
    elasticity_matrix = np.zeros((len(comp_prices), len(prices)))
    delta_p = 5
    
    for i, cp in enumerate(comp_prices):
        for j, p in enumerate(prices):
            ctx = {'competitor_price': cp, 'is_weekend': 0, 'is_festival': 0,
                   'inventory': 100, 'month': 6, 'day_of_week': 2, 'temperature': 30.0}
            d1 = max(0, float(model.predict(pd.DataFrame([{**ctx, 'price': p}])[FEATURES])[0]))
            d2 = max(0, float(model.predict(pd.DataFrame([{**ctx, 'price': p + delta_p}])[FEATURES])[0]))
            e = point_elasticity(p, p + delta_p, d1, d2) if d1 > 0 else 0
            elasticity_matrix[i, j] = np.clip(e, -5, 2)
    
    fig, ax = plt.subplots(figsize=(12, 9))
    sns.heatmap(elasticity_matrix,
        xticklabels=[f'₹{p:.0f}' for p in prices],
        yticklabels=[f'₹{cp:.0f}' for cp in comp_prices],
        cmap='RdYlGn', center=0, vmin=-3, vmax=1,
        annot=True, fmt='.1f', linewidths=0.3,
        ax=ax, annot_kws={'size': 7})
    ax.set_title('Elasticity Heatmap: Our Price (X) vs Competitor Price (Y)\n'
                 'Green = Inelastic (safe to raise) | Red = Elastic (lower to boost revenue)',
                 fontsize=11)
    ax.set_xlabel('Our Price (₹)'); ax.set_ylabel('Competitor Price (₹)')
    ax.set_facecolor('#1a1a2e'); fig.patch.set_facecolor('#0f0f0f')
    plt.tight_layout(); plt.show()
else:
    print('⚠️ Model not found. Run notebook 02 first to train and save the model.')

**Finding:** The heatmap shows where it's safe to raise prices (green/inelastic zones) vs where a price cut would grow revenue (red/elastic zones). When competitors are expensive, customers tolerate our higher prices — an actionable insight.

## 6. Practical Decision Guide — When to Raise vs Lower Price

In [ ]:
scenarios = [
    {'label': 'Festival Weekend',    'is_festival': 1, 'is_weekend': 1, 'competitor_price': 120},
    {'label': 'Normal Weekday',      'is_festival': 0, 'is_weekend': 0, 'competitor_price': 120},
    {'label': 'Competitor Discount', 'is_festival': 0, 'is_weekend': 0, 'competitor_price': 85},
    {'label': 'Low Inventory',       'is_festival': 0, 'is_weekend': 1, 'competitor_price': 130},
]

BASE_PRICE = 110
print(f'{"Scenario":<25} | Elasticity | Classification             | Recommendation')
print('-' * 95)

results = []
if MODEL_PATH.exists():
    for s in scenarios:
        ctx = {**s, 'inventory': 100, 'month': 6, 'day_of_week': 2, 'temperature': 30.0}
        ctx.pop('label')
        d1 = max(0.1, float(model.predict(pd.DataFrame([{**ctx, 'price': BASE_PRICE}])[FEATURES])[0]))
        d2 = max(0.1, float(model.predict(pd.DataFrame([{**ctx, 'price': BASE_PRICE + 10}])[FEATURES])[0]))
        e  = point_elasticity(BASE_PRICE, BASE_PRICE + 10, d1, d2)
        cl = classify_elasticity(e)
        rec = '🔼 Safe to raise price' if e > -1 else '🔽 Lower price to grow revenue'
        print(f'{s["label"]:<25} | {e:>10.3f} | {cl[:28]:<28} | {rec}')
        results.append({'scenario': s['label'], 'elasticity': round(e, 3), 'action': rec})
else:
    print('⚠️ Model not found. Run notebook 02 first.')

**Finding:** Elasticity changes drastically by context. Festival weekends are inelastic (raise price safely). When competitors discount, our product becomes elastic (lower price or lose customers). This is the core value of the dynamic pricing engine.

## 7. Summary

### Q&A
**Q: Is our product price-elastic or inelastic overall?**  
A: Borderline — the average elasticity hovers near –1. Context determines which side it falls on: festivals = inelastic, competitor discounts = elastic.

**Q: When should a small business raise prices?**  
A: During festivals, weekends, or when competitors are priced higher — all inelastic conditions where revenue increases with price.

### Data Analysis Key Findings
- **Average elasticity ≈ –0.5 to –1.5** depending on context (near the elastic/inelastic boundary)
- **Festivals shift demand to inelastic** — customers are less price-sensitive during peak seasons
- **Competitor discounts shift demand to elastic** — price matching becomes critical
- **Elasticity heatmap** provides an actionable pricing matrix for any price/competitor combination

### Insights & Next Steps
- The elasticity signal should be incorporated into the optimizer as a constraint (e.g., only raise price if e > –0.8)
- Proceed to **Notebook 04** to build the Reinforcement Learning pricing agent that learns these patterns automatically